# MacroMind — Data Pipeline
**Contributor: Pan Haoming** — `haoming.p@berkeley.edu`

This notebook covers:
1. USDA FoodData Central API integration
2. Nutrition data schema and preprocessing
3. Batch fetching with caching
4. Price table and data quality analysis

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)
from dotenv import load_dotenv

load_dotenv("../.env")

from src.config import USDA_API_KEY, USDA_CACHE_PATH, PRICE_PER_100G, USDA_NUTRIENT_IDS
from src.data_pipeline import (
    search_ingredient, get_nutrition, batch_fetch_ingredients,
    load_cache, NutritionFacts
)

print("✓ Imports OK")
print(f"USDA API key suffix: ...{USDA_API_KEY[-4:]}")

## 1. USDA FoodData Central API

The [USDA FoodData Central](https://fdc.nal.usda.gov/) database provides macro and micro nutrient data for:
- **SR Legacy**: Standard Reference data (clean, research-grade)
- **Survey (FNDDS)**: What People Eat in America survey data
- **Branded Foods**: Manufacturer-submitted data (noisy, inconsistent)

We prefer **SR Legacy** and **Survey** data types for cleaner macro values.

In [ ]:
# Live API demo — search for chicken breast
result = search_ingredient("chicken breast", USDA_API_KEY)

if result:
    print(f"Found: {result['description']}")
    print(f"FDC ID: {result['fdcId']}")
    print(f"Data type: {result['dataType']}")
    print(f"Published: {result.get('publishedDate', 'N/A')}")
    print(f"\nTop 5 nutrients:")
    for n in result.get('foodNutrients', [])[:5]:
        print(f"  {n.get('nutrientName', '?')}: {n.get('value', '?')} {n.get('unitName', '')}")
else:
    print("No result — using cached data instead")

### Nutrient ID Mapping

USDA assigns numeric IDs to nutrients. We extract the 4 macros by their IDs:

In [ ]:
nutrient_table = pd.DataFrame([
    {"Nutrient": k, "USDA ID": v, "Unit": "kcal" if k == "calories" else "g"}
    for k, v in USDA_NUTRIENT_IDS.items()
])
print("Macro nutrient IDs used for extraction:")
display(nutrient_table)

## 2. NutritionFacts Schema

We normalize all USDA responses into a `NutritionFacts` dataclass with consistent units (per 100g).

In [ ]:
# Load pre-fetched cache (avoids repeated API calls during demo)
cache = load_cache(USDA_CACHE_PATH)
print(f"Cache has {len(cache)} ingredients")

# Build DataFrame from cache
rows = []
for name, data in cache.items():
    nf = NutritionFacts.from_dict(data)
    rows.append({
        "Ingredient": name,
        "Calories": nf.calories_per_100g,
        "Protein (g)": nf.protein_per_100g,
        "Carbs (g)": nf.carbs_per_100g,
        "Fat (g)": nf.fat_per_100g,
        "Price/100g ($)": nf.price_per_100g,
    })

df = pd.DataFrame(rows).set_index("Ingredient")
print(f"\nNutrition facts per 100g:")
display(df.round(1))

## 3. Nutrition Data Exploration

In [ ]:
# Protein content per 100g — top ingredients
protein_df = df['Protein (g)'].sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Protein bar chart
protein_df.head(15).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Protein per 100g (Top 15 Ingredients)', fontsize=12)
axes[0].set_xlabel('Protein (g)')
axes[0].invert_yaxis()

# Calorie density scatter
axes[1].scatter(df['Protein (g)'], df['Calories'], alpha=0.7, s=60, color='coral')
for idx, row in df.iterrows():
    if row['Protein (g)'] > 10 or row['Calories'] > 300:
        axes[1].annotate(idx, (row['Protein (g)'], row['Calories']),
                         fontsize=7, alpha=0.8)
axes[1].set_xlabel('Protein (g per 100g)')
axes[1].set_ylabel('Calories (per 100g)')
axes[1].set_title('Calorie vs Protein Density', fontsize=12)

plt.tight_layout()
plt.savefig('../data/nutrition_exploration.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved: data/nutrition_exploration.png")

## 4. Price Table

For the MVP, ingredient prices are estimated from BLS CPI averages (2024) and typical US supermarket prices.
The Kroger API integration is planned for the full system.

In [ ]:
price_df = pd.DataFrame([
    {"Ingredient": k, "Price per 100g ($)": v, "Price per kg ($)": round(v * 10, 2)}
    for k, v in PRICE_PER_100G.items()
]).sort_values("Price per 100g ($)")

fig, ax = plt.subplots(figsize=(8, 8))
price_df.plot(kind='barh', x='Ingredient', y='Price per 100g ($)', ax=ax, color='seagreen')
ax.set_title('Estimated Price per 100g (USD)', fontsize=12)
ax.set_xlabel('Price (USD)')
plt.tight_layout()
plt.show()

display(price_df)

## 5. Batch Fetch Demo

The `batch_fetch_ingredients` function fetches and caches nutrition data for all ingredients.
On first run it makes API calls; on subsequent runs it reads from cache.

In [ ]:
# All ingredients used in our recipe dataset
ALL_INGREDIENTS = list(cache.keys())  # already cached

print(f"Fetching/loading nutrition data for {len(ALL_INGREDIENTS)} ingredients...")
nutrition_data = batch_fetch_ingredients(ALL_INGREDIENTS)
print(f"\nSuccessfully loaded: {len(nutrition_data)} ingredients")

# Verify a few entries
for name in ['chicken breast', 'brown rice', 'broccoli', 'olive oil']:
    nf = nutrition_data.get(name)
    if nf:
        print(f"  {name}: {nf.calories_per_100g:.0f} kcal, {nf.protein_per_100g:.1f}g protein")

## 6. Data Quality Analysis

**Data type hierarchy:** We prefer `SR Legacy > Survey (FNDDS) > Branded` for macro completeness.

**Known challenges:**
- Branded foods often have incomplete nutrient panels (missing micronutrients)
- Cooked vs. raw discrepancies: USDA measures are per 100g as-consumed. We use **cooked** values for starches (rice, pasta, quinoa) and **raw** for proteins and vegetables.
- Fiber is excluded from total carbs in some USDA entries — we use `nutrient.id=205` (Total Carbohydrates) which includes fiber.

**Missing data handling:** Ingredients not found in USDA default to `0.0` for all macros rather than crashing — downstream scoring handles this gracefully.

In [ ]:
# Check for any zero-calorie entries (potential missing data)
zero_cal = [(name, nf) for name, nf in nutrition_data.items() if nf.calories_per_100g == 0]
if zero_cal:
    print("Ingredients with 0 calories (may need review):")
    for name, nf in zero_cal:
        print(f"  {name}")
else:
    print("✓ All ingredients have calorie data")

# Summary statistics
print(f"\nDataset summary:")
print(f"  Total ingredients: {len(nutrition_data)}")
print(f"  Avg calories/100g: {df['Calories'].mean():.0f}")
print(f"  Avg protein/100g: {df['Protein (g)'].mean():.1f}g")
print(f"  Price range: ${price_df['Price per 100g ($)'].min():.2f} - ${price_df['Price per 100g ($)'].max():.2f}")

## 7. Data Scale Summary

| Component | Scale | Source |
|-----------|-------|--------|
| Nutrition facts | 30 ingredients, 4 macros each | USDA FoodData Central |
| Recipes | 65 recipes | Manually curated (RecipeNLG format) |
| Grocery prices | 30 items | BLS CPI estimates |
| USDA FDC total | ~1.1M food items | FoodData Central API |
| RecipeNLG full | 2.2M recipes | Not used for MVP |

**Preprocessing steps applied:**
1. Filter to `SR Legacy` and `Survey (FNDDS)` data types
2. Extract 4 macros by nutrient ID (203, 204, 205, 208)
3. Normalize all values to per-100g basis
4. Cache to JSON to avoid repeated API calls
5. Fuzzy-match ingredient names at query time (handles plural/variant names)